In [ ]:
import sys
from pathlib import Path

# Add project root so `import shared` works
root = Path.cwd().resolve()
while root != root.parent and not (root / "shared").exists():
    root = root.parent

if not (root / "shared").exists():
    raise RuntimeError("Could not find project root containing 'shared/'")

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from shared.paths import get_experiment_paths
from shared.visualization import show_grid, save_grid

import torch
device = "cuda" if torch.cuda.is_available() else "cpu"

paths = get_experiment_paths("10_face_does_not_exist")
paths


In [ ]:
def latest_ckpt(models_dir: Path):
    ckpts = sorted(models_dir.glob("ckpt_epoch_*.pth"))
    if not ckpts:
        raise FileNotFoundError(f"No ckpt_epoch_*.pth found in {models_dir}")
    return ckpts[-1]


In [ ]:
import torch.nn as nn

class FaceGenerator(nn.Module):
    def __init__(self, z_dim=128, feat=64, img_channels=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(z_dim, feat * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(feat * 8),
            nn.ReLU(True),

            nn.ConvTranspose2d(feat * 8, feat * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(feat * 4),
            nn.ReLU(True),

            nn.ConvTranspose2d(feat * 4, feat * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(feat * 2),
            nn.ReLU(True),

            nn.ConvTranspose2d(feat * 2, feat, 4, 2, 1, bias=False),
            nn.BatchNorm2d(feat),
            nn.ReLU(True),

            nn.ConvTranspose2d(feat, img_channels, 4, 2, 1, bias=False),
            nn.Tanh()
        )

    def forward(self, z):
        return self.net(z)


In [ ]:
ckpt_path = latest_ckpt(paths.models_dir)
ckpt = torch.load(ckpt_path, map_location=device)

z_dim = ckpt.get("z_dim", 128)
feat = ckpt.get("feat", 64)
img_channels = ckpt.get("img_channels", 3)

G = FaceGenerator(z_dim=z_dim, feat=feat, img_channels=img_channels).to(device)
G.load_state_dict(ckpt["G"])
G.eval()

print("Loaded:", ckpt_path.name)
print("z_dim:", z_dim, "| feat:", feat, "| img_channels:", img_channels)


In [ ]:
with torch.no_grad():
    z = torch.randn(64, z_dim, 1, 1, device=device)
    samples = G(z)

show_grid(samples, title=f"10_face_does_not_exist | {ckpt_path.name}", nrow=8)


In [ ]:
save_grid(samples, paths.exp_dir / "latest_face_samples.png", nrow=8)
paths.exp_dir / "latest_face_samples.png"
